# 03: Verify a record removal

**What this demonstrates:** The native `CausalRecordEvaluator` removes an actual record at its read boundary, checks the paired state and reports a signed effect.

**Inputs:** Seeded key and distractor records, followed by summary and hint cases with explicit fixture semantics.

**Outputs:** Native per-trial evidence, signed effects, unadjusted p-values, useful-record decisions, measured episode/time costs and figures. These are record-level results, not graph-edge evidence.

Demo uses 10 pairs per record; paper uses 50. Both execute the same task and intervention path. The run reports its actual time.


## 1. Setup


In [ ]:
RUN_MODE = "demo"  # @param ["demo", "paper"]


In [ ]:
import sys
import json
import hashlib
import random
from dataclasses import dataclass
from pathlib import Path
from time import perf_counter

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != "notebooks":
    NOTEBOOK_DIR = NOTEBOOK_DIR / "notebooks"
assert (NOTEBOOK_DIR / "_utils").is_dir(), "Run from the repository or notebooks directory"
sys.path.insert(0, str(NOTEBOOK_DIR))

from _utils import NotebookConfig, print_config, print_repro_stamp, RunManager
from _utils import setup_plotting, COLORS, plot_quality_vs_overhead
from _utils.probe_integration import ParetoPoint
from reckon.core.episode_context import EpisodeContext
from reckon.core.resources import MemoryBudget
from reckon.memory import CausalMemoryManager, PayloadType
from reckon.metrics import (
    CausalRecordEvaluator,
    MemoryEfficiencyMetrics,
    PerformanceMetrics,
)
from reckon.tasks.base import BaseTask, TaskResult
import matplotlib.pyplot as plt

setup_plotting()
print("Native evaluator and plotting imports ready")


In [ ]:
config = NotebookConfig(
    notebook_name="03_contract_evaluation",
    run_mode=RUN_MODE,
    seed=42,
    seed_count=1,
    budget_slots=100,
    task_name="seeded_record_decision",
    num_episodes=5 if RUN_MODE == "demo" else 20,
    eval_n_trials=10 if RUN_MODE == "demo" else 50,
    custom={"evaluator": "native", "intervention": "first eligible read", "fixture_version": 2},
)
print_config(config)
print_repro_stamp(config)


## 2. Problem and hypotheses

The native evaluator distinguishes a significant change from a useful record. Delta is baseline outcome minus removal outcome. A useful result has a strictly positive delta at or above the configured effect threshold, with p < alpha; a harmful effect keeps its negative sign.

**H1:** Removing the seed-matched key lowers the fixture's success rate.

**H2:** Removing a retrieved distractor leaves success unchanged.

The task retrieves its candidates through the memory manager and then uses only the fields defined by the fixture. This exercises actual removal before the returned records reach the decision. The per-record p-values are unadjusted; the useful fraction is descriptive and does not claim multiple-comparison control. The [evaluation manual](../docs/11-Evaluation-Metrics.md) owns the native semantics and identification limits.


## 3. What is measured

The record-level contrast is

$$\Delta_r = \mathbb{E}[\text{outcome with } r] - \mathbb{E}[\text{outcome after removing } r].$$

Each pair uses the same seed and declared task, budget and policy. The evaluator verifies the selected record state and read ordinal, then checks that removal succeeded. It applies a paired two-sided permutation test. Those checks establish the executed intervention contrast; an identified causal claim still needs its estimand, assumptions and controls.

`is_significant` describes a large enough change in either direction. `is_useful` additionally requires the beneficial direction. These record results do not establish a pairwise graph dependency.


## 4. System mapping

```mermaid
flowchart TB
    Seed["Shared seed and configuration"] --> Base["Fresh task and manager"]
    Seed --> Removal["Fresh task and manager"]
    Base --> Keep["Create records; retain read candidate"]
    Removal --> Prune["Create records; remove read candidate"]
    Keep --> Verify["Match selected state and read ordinal"]
    Prune --> Verify
    Verify --> Outcome["Compare the task outcomes"]
    Outcome --> Evidence["Signed delta, p-value, useful flag and trial evidence"]
```

The evaluator uses the manager's scoped read intervention. The task receives the refreshed query result and credits only returned records.


## 5. Experiment design

The key case contains one key and five distractors. Every trial generates its key colour from that trial's seed, and the task's expected colour comes from the same draw. All six records reach the read boundary, but the decision uses only the key. The budget holds the whole fixture, isolating the intervention from capacity pruning.

The summary case adds a retrieved summary that the decision ignores. The hint case uses a hint-dependent outcome with a seeded fallback. These cases test declared task behavior, not semantic substitution between representations.

Each result uses 10 pairs in demo or 50 in paper, a 0.05 effect threshold and alpha 0.05. The trial-count comparison executes additional pair counts and compares correct classification against the fixture's known useful/distractor labels. It reports the actual episode and elapsed-time cost rather than assuming a quality curve.


## 6. Demo Run


In [ ]:
DELTA_THRESHOLD = config.eval_delta_threshold
ALPHA = config.eval_alpha
N_TRIALS = config.eval_n_trials
print(f"Trials per record: {N_TRIALS}; effect threshold: {DELTA_THRESHOLD}; alpha: {ALPHA}")


In [ ]:
@dataclass
class DecisionConfig:
    scenario: str = "key"


class RecordDecisionTask(BaseTask):
    name = "Seeded record decision"
    description = "Read a seeded key or hint; ignore retrieved distractors"

    def __init__(self, scenario="key"):
        self.config = DecisionConfig(scenario)

    def run(self, cmm, num_episodes, seed, *, reset_memory=False, learn_utility=True):
        outcomes = []
        for episode in range(num_episodes):
            rng = random.Random(seed + episode)
            key_colour = rng.choice(["red", "blue", "green"])
            fallback = rng.random() < 1 / 3
            # Fresh fixture every episode; the intervention survives this reset.
            cmm.set_episode(f"decision_{episode}", reset_memory=True)
            records = {f"distractor_{i}": {"type": "observation", "room": i} for i in range(5)}
            if self.config.scenario == "hint":
                records["indirect_hint"] = {"type": "hint", "content": "Avoid the broken door"}
            else:
                records["key_record"] = {"type": "key", "colour": key_colour}
                if self.config.scenario == "summary":
                    records["summary_record"] = {"type": "summary", "colour": key_colour}
            with EpisodeContext(episode_id=f"decision_{episode}"):
                for label, payload in records.items():
                    cmm.write(payload, PayloadType.CUE, tags=[label], source_op="seeded_fixture")
                retrieved = cmm.retrieve(limit=100)
                if self.config.scenario == "hint":
                    success = any(r.payload["type"] == "hint" for r in retrieved) or fallback
                else:
                    success = any(
                        r.payload["type"] == "key" and r.payload["colour"] == key_colour
                        for r in retrieved
                    )
                if learn_utility:
                    cmm.observe(float(success))
                outcomes.append(float(success))
        return TaskResult(
            PerformanceMetrics(success_rate=sum(outcomes) / len(outcomes)),
            MemoryEfficiencyMetrics(),
            {},
            raw_data={"scenario": self.config.scenario, "outcomes": outcomes},
        )


def memory_factory():
    return CausalMemoryManager(MemoryBudget(max_slots=config.budget_slots))


record_labels = ["key_record"] + [f"distractor_{i}" for i in range(5)]
print("Targets:", record_labels)


In [ ]:
executed_evaluations = []


def evaluate_native(label, n_trials=N_TRIALS, scenario="key", seed=config.seed):
    started = perf_counter()
    evaluator = CausalRecordEvaluator(
        lambda: RecordDecisionTask(scenario),
        memory_factory,
        lambda result: result.performance.success_rate,
        n_trials=n_trials,
        delta_threshold=DELTA_THRESHOLD,
        alpha=ALPHA,
        seed_base=seed,
    )
    result = evaluator.evaluate_record(label, record_selector=lambda record: label in record.tags)
    elapsed_s = perf_counter() - started
    row = {
        **result.to_dict(),
        "scenario": scenario,
        "probe_episodes": sum(2 * trial["episodes_per_arm"] for trial in result.trials),
        "probe_elapsed_s": elapsed_s,
    }
    executed_evaluations.append(row)
    return row


# Every evaluation below uses this native path; no local removal/statistics replacement.
print("Native evaluator ready")


## 7. Full Run


In [ ]:
started = perf_counter()
base_result = RecordDecisionTask().run(memory_factory(), config.num_episodes, config.seed)
base_elapsed_s = perf_counter() - started
BASE_EPISODES = config.num_episodes
all_results = [evaluate_native(label) for label in record_labels]
main_probe_episodes = sum(r["probe_episodes"] for r in all_results)
main_probe_elapsed_s = sum(r["probe_elapsed_s"] for r in all_results)
print(f"Executed base episodes: {BASE_EPISODES}; success: {base_result.performance.success_rate}")
print(f"Main evaluation episodes: {main_probe_episodes}; elapsed: {main_probe_elapsed_s:.6f}s")


## 8. Results


In [ ]:
print(f"{'Target':<18} {'Delta':>8} {'p-value':>10} {'Useful':>8} {'Episodes':>10} {'Seconds':>10}")
for row in all_results:
    print(
        f"{row['record_id']:<18} {row['delta']:8.3f} {row['p_value']:10.4f} {str(row['is_useful']):>8} {row['probe_episodes']:10d} {row['probe_elapsed_s']:10.6f}"
    )
n_useful = sum(r["is_useful"] for r in all_results)
print(f"Useful fraction among these targets: {n_useful}/{len(all_results)}")


In [ ]:
fig_effects, axes = plt.subplots(1, 2, figsize=(12, 5))
labels = [r["record_id"] for r in all_results]
colours = [COLORS["primary"] if r["is_useful"] else COLORS["neutral"] for r in all_results]
axes[0].bar(labels, [r["delta"] for r in all_results], color=colours)
axes[0].axhline(DELTA_THRESHOLD, color="red", linestyle="--", label="Useful effect threshold")
axes[0].axhline(0, color="black", linewidth=0.5)
axes[0].tick_params(axis="x", rotation=45)
axes[0].set(ylabel="Baseline minus removal outcome", title="Signed record effects")
axes[0].legend()
axes[1].scatter([r["delta"] for r in all_results], [r["p_value"] for r in all_results], c=colours)
axes[1].axvline(DELTA_THRESHOLD, color="red", linestyle="--")
axes[1].axhline(ALPHA, color="orange", linestyle="--")
axes[1].set(
    xlabel="Signed delta",
    ylabel="Unadjusted p-value",
    title="Beneficial and significant are separate",
)
fig_effects.tight_layout()
plt.show()


In [ ]:
key_result = next(r for r in all_results if r["record_id"] == "key_record")
h1_pass = key_result["is_useful"] and key_result["delta"] == 1.0
h2_pass = all(
    r["delta"] == 0 and not r["is_useful"]
    for r in all_results
    if r["record_id"].startswith("distractor")
)
assert h1_pass and h2_pass
assert all(r["baseline_metric"] == 1 for r in all_results)
assert all(t["intervention"]["removed"] for r in all_results for t in r["trials"])
print(
    "The trial-matched key succeeds in every baseline; removal changes the outcome. Distractor removal does not."
)


### Measured trial-count trade-off

The base task above really ran. Probe episode counts include both arms, and elapsed time includes evaluator setup, execution and statistics. Episode fraction and elapsed-time fraction are reported separately; neither is called a generic fraction of compute.

For each trial count, classification accuracy compares `is_useful` against this fixture's known labels. This is a small controlled example of detection power and cost, not a recommended trial count for another workload. The main trial count reuses its already measured results.


In [ ]:
trial_counts = [5, 10] if RUN_MODE == "demo" else [5, 10, 20, 50]
tradeoff_results = []
for count in trial_counts:
    results = (
        all_results
        if count == N_TRIALS
        else [evaluate_native(label, count) for label in record_labels]
    )
    correct = sum(r["is_useful"] == (r["record_id"] == "key_record") for r in results)
    probe_episodes = sum(r["probe_episodes"] for r in results)
    probe_s = sum(r["probe_elapsed_s"] for r in results)
    tradeoff_results.append(
        {
            "n_trials": count,
            "classification_accuracy": correct / len(results),
            "probe_episodes": probe_episodes,
            "probe_elapsed_s": probe_s,
            "probe_episode_fraction": probe_episodes / (BASE_EPISODES + probe_episodes),
            "probe_time_fraction": probe_s / (base_elapsed_s + probe_s),
        }
    )
print(json.dumps(tradeoff_results, indent=2))


In [ ]:
pareto_points = [
    ParetoPoint(
        x=row["probe_episodes"],
        y=row["classification_accuracy"],
        label=f"n={row['n_trials']}",
        metadata=row,
    )
    for row in tradeoff_results
]
for point in pareto_points:
    point.is_pareto = not any(
        other.x <= point.x and other.y >= point.y and (other.x < point.x or other.y > point.y)
        for other in pareto_points
    )
fig_cost, ax = plt.subplots(figsize=(9, 5))
plot_quality_vs_overhead(
    ax,
    pareto_points,
    xlabel="Executed probe episodes (both arms)",
    ylabel="Correct useful/distractor classification",
    title="Measured cost and classification in this fixture",
)
fig_cost.tight_layout()
plt.show()
print(
    "Frontier trial counts for this fixture:",
    [p.metadata["n_trials"] for p in pareto_points if p.is_pareto],
)


### Surface Appearance and Removal Outcomes

Two synthetic cases contrast what looks useful with what the fixture reads. They do not establish general causal identification or interchangeable information representations.


In [ ]:
summary_eval = evaluate_native("summary_record", scenario="summary", seed=config.seed + 100)
assert summary_eval["delta"] == 0 and not summary_eval["is_useful"]
print("Summary case:", {k: summary_eval[k] for k in ("delta", "p_value", "is_useful")})
print(
    "The summary reaches retrieval but the fixture's decision uses only type=key. This does not test semantic substitution."
)


In [ ]:
hint_eval = evaluate_native("indirect_hint", scenario="hint", seed=config.seed + 200)
print(
    "Hint case:",
    {
        k: hint_eval[k]
        for k in ("baseline_metric", "intervention_metric", "delta", "p_value", "is_useful")
    },
)
print(
    "The fixture defines hint presence as success; without it, the seed-matched fallback succeeds with probability 1/3."
)


**The lesson:** A record's effect depends on the task's decision. The retrieved summary has no effect because the decision ignores it; the hint fixture explicitly depends on its hint. Both now run through the same native evaluator and retain their trial evidence. Neither identifies a causal relation in a wider system.

A MEASURED debugger edge still needs source-target-bound evidence under the [debugger contract](../docs/10-Debugger-Visualization.md). Record-level trial results do not provide that edge automatically.


## 9. Inspecting the evidence

The export includes each paired seed, actual record IDs, selected-state digest, read position, removal status and outcomes in `evaluations.json`. Inspect those before interpreting a delta. It also saves measured cost and trial-count tables plus figures.

This notebook does not package the managers' event streams into a debugger run or emit pair-bound graph probes. For a separately exported native event run, follow the [debugger quick start](../src/reckon/viz/README.md#quick-start).


## 10. Limits and next tests

This is a synthetic task with known labels, a small record set and fixed effect/significance thresholds. It exercises the native evaluator; it does not use Prober's adaptive or multiple-comparison correction. Per-record tests are unadjusted, and classification accuracy describes this fixture only. Measured timing is specific to this run and machine.

Next comparisons belong on real MiniGrid episodes, threshold/alpha sensitivity, separately labelled gradient proxies and larger record sets. Keep task episodes, both probe arms, raw signed outcomes and full probe time together in each comparison.


## Export Artifacts


In [ ]:
run = RunManager(config)
run.save_config()
all_probe_episodes = sum(row["probe_episodes"] for row in executed_evaluations)
all_probe_s = sum(row["probe_elapsed_s"] for row in executed_evaluations)
execution_totals = {
    "base_episodes": BASE_EPISODES,
    "base_elapsed_s": base_elapsed_s,
    "probe_episodes": all_probe_episodes,
    "probe_elapsed_s": all_probe_s,
    "total_episodes": BASE_EPISODES + all_probe_episodes,
    "total_measured_s": base_elapsed_s + all_probe_s,
    "probe_episode_fraction": all_probe_episodes / (BASE_EPISODES + all_probe_episodes),
    "probe_time_fraction": all_probe_s / (base_elapsed_s + all_probe_s),
    "evaluation_calls": len(executed_evaluations),
}
for filename, value in [
    ("evaluations.json", executed_evaluations),
    ("probe_costs.json", execution_totals),
    ("trial_counts.json", tradeoff_results),
    ("pareto_frontier.json", [point.to_dict() for point in pareto_points]),
]:
    (run.run_dir / filename).write_text(json.dumps(value, indent=2), encoding="utf-8")
run.save_metrics([{k: v for k, v in row.items() if k != "trials"} for row in all_results])
run.save_figure(fig_effects, "ablation_deltas")
run.save_figure(fig_cost, "classification_vs_probe_episodes")
plt.close("all")
notebook_source = json.loads((NOTEBOOK_DIR / "03_contract_evaluation_executable.ipynb").read_text())
source_digest = hashlib.sha256(
    json.dumps([cell["source"] for cell in notebook_source["cells"]], sort_keys=True).encode()
).hexdigest()
(run.run_dir / "source.json").write_text(
    json.dumps(
        {
            "native_evaluator_commit": config.git_hash,
            "notebook_cells_sha256": source_digest,
            "notebook_source_uncommitted": bool(
                __import__("subprocess")
                .run(
                    [
                        "git",
                        "status",
                        "--porcelain",
                        "--",
                        "notebooks/03_contract_evaluation_executable.ipynb",
                    ],
                    cwd=NOTEBOOK_DIR.parent,
                    capture_output=True,
                    text=True,
                    check=True,
                )
                .stdout
            ),
        },
        indent=2,
    )
)
run.write_summary(
    {
        "main_targets": len(all_results),
        "useful_targets": n_useful,
        "key_delta": key_result["delta"],
        "key_p_value": key_result["p_value"],
        "summary_delta": summary_eval["delta"],
        "hint_delta": hint_eval["delta"],
        "h1_supported": h1_pass,
        "h2_supported": h2_pass,
        **execution_totals,
    },
    notes="Native evaluator; single-instance ablation. Per-record p-values are unadjusted. Counts include the main, additional trial-count, summary and hint evaluations once each. Figure/export time is outside measured task and probe time.",
)
run.print_artifacts()
print("Run bundle:", run.run_dir)
print(json.dumps(execution_totals, indent=2))
